In [4]:
# 0. Instalamos las herramientas necesarias
!pip install pymssql fpdf pytz matplotlib

import pandas as pd
from fpdf import FPDF
from datetime import datetime
import pytz
from google.colab import userdata
import pymssql
import warnings
import matplotlib.pyplot as plt
import numpy as np
import os

# Ocultar advertencias amarillas de Pandas
warnings.filterwarnings('ignore', category=UserWarning)

# ==============================================================
# 📅 1. DEFINE EL PERÍODO DEL REPORTE AQUÍ
# ==============================================================
FECHA_INICIO = '2026-10-13'
FECHA_FIN    = '2026-10-13'
# ==============================================================

print(f"Conectando a Wiidem para extraer datos del {FECHA_INICIO} al {FECHA_FIN}...")

try:
    conexion = pymssql.connect(
        server='200.49.98.250',
        user=userdata.get('WIIDEM_USER'),
        password=userdata.get('WIIDEM_PASS'),
        database=userdata.get('WIIDEM_DB'),
        login_timeout=15
    )

    query_oee = f"SELECT Cell AS MAQUINA, AVG(Oee) AS OEE, AVG(Availability) AS DISPONIBILIDAD, AVG(Performance) AS PERFORMANCE, AVG(Quality) AS CALIDAD FROM PROD_D_03 WHERE Date >= '{FECHA_INICIO}' AND Date <= '{FECHA_FIN}' AND Cell LIKE 'CEL%' GROUP BY Cell ORDER BY Cell"
    df_oee = pd.read_sql(query_oee, conexion)

    query_prod_diaria = f"SELECT CAST(Date AS DATE) AS FECHA, Cell AS MAQUINA, Turn AS TURNO, ProductiveTime AS PRODUCTIVO, DownTime AS PARADA FROM PROD_D_02 WHERE Date >= '{FECHA_INICIO}' AND Date <= '{FECHA_FIN}' AND Cell LIKE 'CEL%'"
    df_prod_diaria = pd.read_sql(query_prod_diaria, conexion)

    query_horarios_diarios = f"SELECT CAST(Started AS DATE) AS FECHA, Cell AS MAQUINA, Turn AS TURNO, MIN(Started) AS HORA_INICIO, MAX(Finish) AS HORA_CIERRE FROM EVENT_01 WHERE Started >= '{FECHA_INICIO}' AND Started <= '{FECHA_FIN} 23:59:59' AND Cell LIKE 'CEL%' GROUP BY CAST(Started AS DATE), Cell, Turn"
    df_horarios_diarios = pd.read_sql(query_horarios_diarios, conexion)

    query_resumen_tiempos = f"SELECT UPPER(et1.Name) AS CategoriaPadre, UPPER(et2.Name) AS CategoriaHijo, SUM(e.Interval) AS MinutosTotales FROM EVENT_01 e LEFT JOIN EventType et1 ON e.EventTypeLevel1 = et1.EventTypeId LEFT JOIN EventType et2 ON e.EventTypeLevel2 = et2.EventTypeId WHERE e.Started >= '{FECHA_INICIO}' AND e.Started <= '{FECHA_FIN} 23:59:59' AND e.Cell LIKE 'CEL%' GROUP BY et1.Name, et2.Name"
    df_resumen_tiempos = pd.read_sql(query_resumen_tiempos, conexion)

    query_top_fallas = f"SELECT TOP 15 COALESCE(et3.Name, et2.Name, et1.Name) AS NombreFalla, SUM(e.Interval) AS MinutosPerdidos FROM EVENT_01 e LEFT JOIN EventType et1 ON e.EventTypeLevel1 = et1.EventTypeId LEFT JOIN EventType et2 ON e.EventTypeLevel2 = et2.EventTypeId LEFT JOIN EventType et3 ON e.EventTypeLevel3 = et3.EventTypeId WHERE e.Started >= '{FECHA_INICIO}' AND e.Started <= '{FECHA_FIN} 23:59:59' AND e.Cell LIKE 'CEL%' AND (et1.Name LIKE '%FALLA%' OR et1.Name LIKE '%GESTION%') AND COALESCE(et2.Name, '') NOT LIKE '%REFRIGERIO%' AND COALESCE(et2.Name, '') NOT LIKE '%BAÑO%' AND COALESCE(et2.Name, '') NOT LIKE '%DESCANSO%' GROUP BY COALESCE(et3.Name, et2.Name, et1.Name) ORDER BY MinutosPerdidos DESC"
    df_top_fallas = pd.read_sql(query_top_fallas, conexion)

    query_tendencia = f"SELECT e.Cell AS MAQUINA, DATEPART(HOUR, e.Started) AS HORA, SUM(e.Interval) AS MinutosPerdidos FROM EVENT_01 e LEFT JOIN EventType et1 ON e.EventTypeLevel1 = et1.EventTypeId LEFT JOIN EventType et2 ON e.EventTypeLevel2 = et2.EventTypeId WHERE e.Started >= '{FECHA_INICIO}' AND e.Started <= '{FECHA_FIN} 23:59:59' AND e.Cell LIKE 'CEL%' AND (et1.Name LIKE '%FALLA%' OR et1.Name LIKE '%GESTION%') AND COALESCE(et2.Name, '') NOT LIKE '%REFRIGERIO%' AND COALESCE(et2.Name, '') NOT LIKE '%BAÑO%' GROUP BY e.Cell, DATEPART(HOUR, e.Started) ORDER BY HORA"
    df_tendencia = pd.read_sql(query_tendencia, conexion)

    query_resumen_maq = f"SELECT e.Cell AS MAQUINA, UPPER(et1.Name) AS CategoriaPadre, UPPER(et2.Name) AS CategoriaHijo, SUM(e.Interval) AS MinutosTotales FROM EVENT_01 e LEFT JOIN EventType et1 ON e.EventTypeLevel1 = et1.EventTypeId LEFT JOIN EventType et2 ON e.EventTypeLevel2 = et2.EventTypeId WHERE e.Started >= '{FECHA_INICIO}' AND e.Started <= '{FECHA_FIN} 23:59:59' AND e.Cell LIKE 'CEL%' GROUP BY e.Cell, et1.Name, et2.Name"
    df_resumen_maq = pd.read_sql(query_resumen_maq, conexion)

    query_top3_maquina = f"WITH FallasRankeadas AS (SELECT e.Cell AS MAQUINA, COALESCE(et3.Name, et2.Name, et1.Name) AS NombreFalla, SUM(e.Interval) AS Minutos, ROW_NUMBER() OVER(PARTITION BY e.Cell ORDER BY SUM(e.Interval) DESC) as RankFalla FROM EVENT_01 e LEFT JOIN EventType et1 ON e.EventTypeLevel1 = et1.EventTypeId LEFT JOIN EventType et2 ON e.EventTypeLevel2 = et2.EventTypeId LEFT JOIN EventType et3 ON e.EventTypeLevel3 = et3.EventTypeId WHERE e.Started >= '{FECHA_INICIO}' AND e.Started <= '{FECHA_FIN} 23:59:59' AND e.Cell LIKE 'CEL%' AND (et1.Name LIKE '%FALLA%' OR et1.Name LIKE '%GESTION%') AND COALESCE(et2.Name, '') NOT LIKE '%REFRIGERIO%' AND COALESCE(et2.Name, '') NOT LIKE '%BAÑO%' AND COALESCE(et2.Name, '') NOT LIKE '%DESCANSO%' GROUP BY e.Cell, COALESCE(et3.Name, et2.Name, et1.Name)) SELECT MAQUINA, NombreFalla, Minutos FROM FallasRankeadas WHERE RankFalla <= 3"
    df_top3_maquina = pd.read_sql(query_top3_maquina, conexion)

    query_dt_area = f"SELECT e.Cell AS MAQUINA, COALESCE(et3.Name, et2.Name, et1.Name) AS NombreFalla, et1.Name AS CategoriaPadre, SUM(e.Interval) AS Minutos FROM EVENT_01 e LEFT JOIN EventType et1 ON e.EventTypeLevel1 = et1.EventTypeId LEFT JOIN EventType et2 ON e.EventTypeLevel2 = et2.EventTypeId LEFT JOIN EventType et3 ON e.EventTypeLevel3 = et3.EventTypeId WHERE e.Started >= '{FECHA_INICIO}' AND e.Started <= '{FECHA_FIN} 23:59:59' AND e.Cell LIKE 'CEL%' AND (et1.Name LIKE '%FALLA%' OR et1.Name LIKE '%GESTION%') AND COALESCE(et2.Name, '') NOT LIKE '%REFRIGERIO%' AND COALESCE(et2.Name, '') NOT LIKE '%BAÑO%' AND COALESCE(et2.Name, '') NOT LIKE '%DESCANSO%' GROUP BY e.Cell, et1.Name, COALESCE(et3.Name, et2.Name, et1.Name)"
    df_dt_area = pd.read_sql(query_dt_area, conexion)

    # NUEVA CONSULTA 10: TOP 5 PRODUCTOS POR MÁQUINA
    query_top_prod = f"""
    WITH ProductoRankeado AS (
        SELECT
            Cell AS MAQUINA,
            Code AS CODIGO_PRODUCTO,
            SUM(Good) AS BUENAS,
            SUM(Rework) AS RETRABAJO,
            SUM(Scrap) AS OBSERVADAS,
            SUM(Good + Rework + Scrap) AS TOTAL_PRODUCIDO,
            ROW_NUMBER() OVER(PARTITION BY Cell ORDER BY SUM(Good + Rework + Scrap) DESC) as RankProd
        FROM PROD_D_01
        WHERE Date >= '{FECHA_INICIO}' AND Date <= '{FECHA_FIN}'
          AND Cell LIKE 'CEL%'
        GROUP BY Cell, Code
    )
    SELECT * FROM ProductoRankeado WHERE RankProd <= 5
    """
    df_top_prod = pd.read_sql(query_top_prod, conexion)

    conexion.close()
    print("✅ Datos extraídos exitosamente. Procesando y generando reportes...")

    # ---------------------------------------------------------
    # PROCESAMIENTO
    # ---------------------------------------------------------
    def formatear_nombre_maq(nombre):
        return str(nombre).replace('CELDA ', 'Celda 0').replace('CEL', 'Celda ') + ' Fumis'

    def min_a_hhmm_hs(minutos_num):
        if pd.isna(minutos_num) or minutos_num <= 0: return "00:00 hs"
        hrs = int(minutos_num // 60)
        mins = int(minutos_num % 60)
        return f"{hrs:02d}:{mins:02d} hs"

    df_oee['MAQUINA'] = df_oee['MAQUINA'].apply(formatear_nombre_maq)
    cols_kpi = ['OEE', 'DISPONIBILIDAD', 'PERFORMANCE', 'CALIDAD']
    for col in cols_kpi: df_oee[col] = pd.to_numeric(df_oee[col], errors='coerce').fillna(0.0)
    total_grupo = df_oee[cols_kpi].mean() if not df_oee.empty else None

    no_reg_dict = {}
    apertura_mensual_dict = {}

    if not df_prod_diaria.empty:
        df_prod_diaria['FECHA'] = pd.to_datetime(df_prod_diaria['FECHA'])
        df_horarios_diarios['FECHA'] = pd.to_datetime(df_horarios_diarios['FECHA'])
        df_diario = pd.merge(df_prod_diaria, df_horarios_diarios, on=['FECHA', 'MAQUINA', 'TURNO'], how='left')
        df_diario['MAQUINA'] = df_diario['MAQUINA'].apply(formatear_nombre_maq)
        df_diario['MIN_APERTURA_NETA'] = df_diario['PRODUCTIVO'].fillna(0) + df_diario['PARADA'].fillna(0)

        df_apertura_mensual = df_diario.groupby('MAQUINA')['MIN_APERTURA_NETA'].sum().reset_index()
        for _, row in df_apertura_mensual.iterrows(): apertura_mensual_dict[row['MAQUINA']] = row['MIN_APERTURA_NETA']

        df_diario['HORA_INICIO_DT'] = pd.to_datetime(df_diario['HORA_INICIO'])
        df_diario['HORA_CIERRE_DT'] = pd.to_datetime(df_diario['HORA_CIERRE'])
        df_diario['DURACION_TOTAL'] = (df_diario['HORA_CIERRE_DT'] - df_diario['HORA_INICIO_DT']).dt.total_seconds() / 60.0
        df_diario['MIN_NO_REGISTRADO'] = df_diario['DURACION_TOTAL'] - df_diario['MIN_APERTURA_NETA']
        df_diario.loc[df_diario['MIN_NO_REGISTRADO'] < 0, 'MIN_NO_REGISTRADO'] = 0
        df_diario['INICIO_MIN'] = df_diario['HORA_INICIO_DT'].dt.hour * 60 + df_diario['HORA_INICIO_DT'].dt.minute
        df_diario['CIERRE_MIN'] = df_diario['HORA_CIERRE_DT'].dt.hour * 60 + df_diario['HORA_CIERRE_DT'].dt.minute

        df_turnos = df_diario.groupby(['TURNO', 'MAQUINA']).agg({'INICIO_MIN': 'median', 'CIERRE_MIN': 'median', 'MIN_APERTURA_NETA': 'mean', 'MIN_NO_REGISTRADO': 'mean'}).reset_index()
        def format_hora_habitual(mins): return "-" if pd.isna(mins) else f"{int(mins // 60) % 24:02d}:{int(mins % 60):02d}"
        df_turnos['TXT_INICIO'], df_turnos['TXT_CIERRE'] = df_turnos['INICIO_MIN'].apply(format_hora_habitual), df_turnos['CIERRE_MIN'].apply(format_hora_habitual)
        df_turnos['TXT_APERTURA_NETA'], df_turnos['TXT_NO_REGISTRADO'] = df_turnos['MIN_APERTURA_NETA'].apply(min_a_hhmm_hs), df_turnos['MIN_NO_REGISTRADO'].apply(min_a_hhmm_hs)

        df_noreg = df_diario.groupby('MAQUINA')['MIN_NO_REGISTRADO'].sum().reset_index()
        for _, row in df_noreg.iterrows(): no_reg_dict[row['MAQUINA']] = row['MIN_NO_REGISTRADO']
    else: df_turnos = pd.DataFrame()

    tiempos_agrupados = {'Produccion': 0, 'Fallas/Gestion': 0, 'Paradas Prog.': 0, 'Proyecto': 0, 'Descansos': 0}
    if not df_resumen_tiempos.empty:
        for _, row in df_resumen_tiempos.iterrows():
            cat, subcat = str(row['CategoriaPadre']).upper(), str(row['CategoriaHijo']).upper()
            mins = row['MinutosTotales'] if not pd.isna(row['MinutosTotales']) else 0
            if 'REFRIGERIO' in subcat or 'BAÑO' in subcat or 'DESCANSO' in subcat: tiempos_agrupados['Descansos'] += mins
            elif 'PROYECTO' in subcat: tiempos_agrupados['Proyecto'] += mins
            elif 'PRODUCCION' in cat: tiempos_agrupados['Produccion'] += mins
            elif 'PARADA PROGRAMADA' in cat: tiempos_agrupados['Paradas Prog.'] += mins
            elif 'PROYECTO' in cat: tiempos_agrupados['Proyecto'] += mins
            elif cat != 'NONE' and cat != 'NAN': tiempos_agrupados['Fallas/Gestion'] += mins

    resumen_maq_dict = {}
    if not df_resumen_maq.empty:
        df_resumen_maq['MAQUINA'] = df_resumen_maq['MAQUINA'].apply(formatear_nombre_maq)
        for _, row in df_resumen_maq.iterrows():
            m = row['MAQUINA']
            cat, subcat = str(row['CategoriaPadre']).upper(), str(row['CategoriaHijo']).upper()
            mins = row['MinutosTotales'] if not pd.isna(row['MinutosTotales']) else 0
            if m not in resumen_maq_dict: resumen_maq_dict[m] = {'Produccion': 0, 'Fallas': 0, 'ParadaProg': 0, 'Descanso': 0}

            if 'REFRIGERIO' in subcat or 'BAÑO' in subcat or 'DESCANSO' in subcat: resumen_maq_dict[m]['Descanso'] += mins
            elif 'PRODUCCION' in cat: resumen_maq_dict[m]['Produccion'] += mins
            elif 'PARADA PROGRAMADA' in cat: resumen_maq_dict[m]['ParadaProg'] += mins
            elif 'PROYECTO' not in cat and 'PROYECTO' not in subcat and cat != 'NONE' and cat != 'NAN':
                resumen_maq_dict[m]['Fallas'] += mins

    top3_dict = {}
    if not df_top3_maquina.empty:
        df_top3_maquina['MAQUINA'] = df_top3_maquina['MAQUINA'].apply(formatear_nombre_maq)
        for _, row in df_top3_maquina.iterrows():
            m = row['MAQUINA']
            if m not in top3_dict: top3_dict[m] = []
            falla = str(row['NombreFalla'])[:38]
            mins = int(row['Minutos']) if not pd.isna(row['Minutos']) else 0
            top3_dict[m].append(f"- {falla} ({mins}m)")

    areas_dt_dict = {}
    if not df_dt_area.empty:
        for _, row in df_dt_area.iterrows():
            maq = formatear_nombre_maq(row['MAQUINA'])
            falla = str(row['NombreFalla']).upper()
            padre = str(row['CategoriaPadre']).upper()
            mins = float(row['Minutos']) if not pd.isna(row['Minutos']) else 0.0

            if maq not in areas_dt_dict:
                areas_dt_dict[maq] = {'MANTEN.': 0, 'DISPOSIT.': 0, 'TECNOL.': 0, 'LOGIST.': 0, 'GESTION': 0, 'OTROS': 0}

            if 'DISPOSITIVO' in falla or 'SENSOR' in falla or 'BRIDA' in falla or 'CALIBRE' in falla or 'AJUSTE' in falla:
                areas_dt_dict[maq]['DISPOSIT.'] += mins
            elif 'TECNOLOG' in falla or 'ROBOT' in falla or 'TRAYECTORIA' in falla or 'AUTOMATIZACI' in falla:
                areas_dt_dict[maq]['TECNOL.'] += mins
            elif 'LOGIST' in falla or 'ABAST' in falla or 'COMPONENTE' in falla or 'MATERIA PRIMA' in falla or 'ESTIBADO' in falla or 'L13' in falla:
                areas_dt_dict[maq]['LOGIST.'] += mins
            elif 'MANTENIMIENTO' in falla or 'NAVE NUEVA' in falla or 'PICO' in falla or 'ESPESOR' in falla or 'M2' in falla:
                areas_dt_dict[maq]['MANTEN.'] += mins
            elif 'GESTION' in padre or 'GESTION' in falla or 'CALIDAD' in falla:
                areas_dt_dict[maq]['GESTION'] += mins
            else:
                areas_dt_dict[maq]['MANTEN.'] += mins

    if not df_top_prod.empty:
        df_top_prod['MAQUINA'] = df_top_prod['MAQUINA'].apply(formatear_nombre_maq)

    # ---------------------------------------------------------
    # MATPLOTLIB - GRÁFICOS
    # ---------------------------------------------------------
    if not df_oee.empty:
        try:
            maquinas = df_oee['MAQUINA'].tolist()
            fig, ax = plt.subplots(figsize=(10, 4.5))
            x, width = np.arange(len(cols_kpi)), 0.8 / len(maquinas)
            colores = plt.cm.tab20(np.linspace(0, 1, len(maquinas)))
            for i, maquina in enumerate(maquinas):
                rects = ax.bar(x + (i - len(maquinas)/2) * width + width/2, df_oee[df_oee['MAQUINA'] == maquina][cols_kpi].values[0], width, label=maquina, color=colores[i])
                ax.bar_label(rects, fmt='%.1f', padding=2, rotation=90, fontsize=8, color='black')
            ax.set_ylabel('Porcentaje (%)'); ax.set_xticks(x); ax.set_xticklabels(cols_kpi); ax.set_ylim(0, df_oee[cols_kpi].max().max() * 1.4)
            ax.legend(bbox_to_anchor=(0.5, 1.20), loc='upper center', ncol=4, frameon=False, fontsize=8.5)
            ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False); ax.grid(axis='y', linestyle='--', alpha=0.5)
            plt.tight_layout(); plt.savefig('grafico_kpis.png', dpi=300, bbox_inches='tight'); plt.close()
        except: pass

    if not df_top_fallas.empty:
        try:
            df_top_fallas = df_top_fallas.sort_values('MinutosPerdidos', ascending=True)
            fig, ax = plt.subplots(figsize=(7, 5))
            barras = ax.barh(df_top_fallas['NombreFalla'].str[:40], df_top_fallas['MinutosPerdidos'], color='#E67E22')
            for barra in barras: ax.text(barra.get_width() + 5, barra.get_y() + barra.get_height()/2, f"{barra.get_width():.0f}m", va='center', ha='left', fontsize=9, color='black')
            ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False); ax.spines['bottom'].set_visible(False); ax.set_xticks([]); ax.tick_params(axis='y', labelsize=8)
            plt.tight_layout(); plt.savefig('grafico_top_fallas.png', dpi=300, bbox_inches='tight'); plt.close()
        except: pass

    if not df_tendencia.empty:
        try:
            df_tendencia['MAQUINA'] = df_tendencia['MAQUINA'].apply(formatear_nombre_maq)
            pivot_tendencia = df_tendencia.pivot(index='HORA', columns='MAQUINA', values='MinutosPerdidos').fillna(0)
            fig, ax = plt.subplots(figsize=(7, 5))
            for col in pivot_tendencia.columns: ax.plot(pivot_tendencia.index, pivot_tendencia[col], marker='o', markersize=4, label=col, linewidth=1.5)
            ax.set_xticks(pivot_tendencia.index); ax.set_xticklabels([f"{int(h):02d}:00" for h in pivot_tendencia.index], fontsize=8, rotation=45); ax.set_ylabel('Min', fontsize=9); ax.tick_params(axis='y', labelsize=8)
            ax.legend(bbox_to_anchor=(0.5, 1.25), loc='upper center', ncol=2, frameon=False, fontsize=8)
            ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False); ax.grid(color='lightgray', linestyle='-', linewidth=0.5, alpha=0.7)
            plt.tight_layout(); plt.savefig('grafico_tendencia.png', dpi=300, bbox_inches='tight'); plt.close()
        except: pass

    # ---------------------------------------------------------
    # GENERACIÓN DEL PDF
    # ---------------------------------------------------------
    COLOR_TITULO, COLOR_GRIS, COLOR_NEGRO = (211, 84, 0), (100, 100, 100), (0, 0, 0)
    TARGETS = {'OEE': 72.0, 'DISPONIBILIDAD': 90.0, 'PERFORMANCE': 80.0, 'CALIDAD': 99.5}

    def obtener_color(valor, kpi_name):
        meta = TARGETS.get(kpi_name, 0)
        if pd.isna(valor): return COLOR_NEGRO
        return (40, 167, 69) if valor >= meta else (230, 126, 34) if valor >= (meta - 10) else (220, 53, 69)

    class ReportePDF(FPDF):
        def print_kpi(self, etiqueta, valor_num, kpi_name, separador=" | "):
            color_dinamico = obtener_color(valor_num, kpi_name)
            valor_txt = f"{valor_num:.1f}%" if not pd.isna(valor_num) else "0.0%"
            self.set_font("Arial", 'B', 10); self.set_text_color(*COLOR_NEGRO); self.cell(self.get_string_width(etiqueta), 6, txt=etiqueta, ln=0)
            self.set_text_color(*color_dinamico); self.cell(self.get_string_width(valor_txt), 6, txt=valor_txt, ln=0)
            self.set_text_color(*COLOR_NEGRO); self.cell(self.get_string_width(separador), 6, txt=separador, ln=0)

    pdf = ReportePDF(orientation='P', unit='mm', format='A4')
    pdf.set_auto_page_break(auto=True, margin=15)

    # --- PÁGINA 1 ---
    pdf.add_page(); pdf.set_font("Arial", 'B', 14); pdf.set_text_color(*COLOR_TITULO)
    pdf.cell(120, 8, txt="SECCIÓN GRUPO: CELDA RENAULT", ln=0)
    fecha_actual = datetime.now(pytz.timezone('America/Argentina/Cordoba')).strftime("%d/%m/%Y %H:%M")
    pdf.set_font("Arial", 'I', 9); pdf.set_text_color(*COLOR_GRIS)
    pdf.cell(0, 8, txt=f"Generado el: {fecha_actual}", ln=1, align='R')
    pdf.set_draw_color(150, 150, 150); pdf.line(10, pdf.get_y(), 200, pdf.get_y()); pdf.ln(4)

    pdf.set_font("Arial", 'B', 12); pdf.set_text_color(*COLOR_TITULO)
    pdf.cell(0, 8, txt=f"1. Resumen OEE del Grupo (Período: {FECHA_INICIO} al {FECHA_FIN})", ln=True)
    pdf.set_draw_color(*COLOR_TITULO); pdf.line(10, pdf.get_y(), 200, pdf.get_y()); pdf.ln(2)
    pdf.set_font("Arial", 'B', 9); pdf.set_text_color(*COLOR_GRIS); pdf.cell(0, 6, txt="Objetivos (Target) del mes: OEE: 72.0% | Disp: 90.0% | Perf: 80.0% | Cal: 99.5%", ln=True); pdf.ln(2)

    if not df_oee.empty:
        pdf.set_font("Arial", 'B', 10); pdf.set_text_color(*COLOR_NEGRO)
        pdf.cell(pdf.get_string_width("Total CELDA RENAULT | "), 6, txt="Total CELDA RENAULT | ", ln=0)
        pdf.print_kpi("OEE: ", total_grupo['OEE'], 'OEE'); pdf.print_kpi("Disp: ", total_grupo['DISPONIBILIDAD'], 'DISPONIBILIDAD'); pdf.print_kpi("Perf: ", total_grupo['PERFORMANCE'], 'PERFORMANCE'); pdf.print_kpi("Cal: ", total_grupo['CALIDAD'], 'CALIDAD', separador=""); pdf.ln(7)
        for _, fila in df_oee.iterrows():
            pdf.cell(5, 5, txt="", ln=0); txt_nombre = f"> {fila['MAQUINA']} | "
            pdf.set_font("Arial", 'B', 9.5); pdf.set_text_color(*COLOR_NEGRO); pdf.cell(pdf.get_string_width(txt_nombre), 5, txt=txt_nombre, ln=0)
            pdf.print_kpi("OEE: ", fila['OEE'], 'OEE'); pdf.print_kpi("Disp: ", fila['DISPONIBILIDAD'], 'DISPONIBILIDAD'); pdf.print_kpi("Perf: ", fila['PERFORMANCE'], 'PERFORMANCE'); pdf.print_kpi("Cal: ", fila['CALIDAD'], 'CALIDAD', separador=""); pdf.ln(5)

        pdf.ln(6); pdf.set_font("Arial", 'B', 12); pdf.set_text_color(*COLOR_TITULO)
        pdf.cell(0, 8, txt="2. Comparativa de KPIs entre Máquinas", ln=True)
        pdf.set_draw_color(*COLOR_TITULO); pdf.line(10, pdf.get_y(), 200, pdf.get_y()); pdf.ln(4)
        if os.path.exists('grafico_kpis.png'): pdf.image('grafico_kpis.png', x=10, w=190)

    # --- PÁGINA 2 ---
    pdf.add_page()
    if not df_turnos.empty:
        pdf.set_font("Arial", 'B', 12); pdf.set_text_color(*COLOR_TITULO)
        pdf.cell(0, 8, txt="3. Horarios y Tiempo de Apertura (Promedio Diario)", ln=True)
        pdf.set_draw_color(*COLOR_TITULO); pdf.line(10, pdf.get_y(), 200, pdf.get_y()); pdf.ln(4)

        w_maq, w_turno, w_inicio, w_cierre, w_apertura, w_noregis = 45, 20, 30, 30, 32, 33
        pdf.set_fill_color(211, 84, 0); pdf.set_text_color(255, 255, 255); pdf.set_draw_color(255, 255, 255); pdf.set_font("Arial", 'B', 8.5)
        for titulo, ancho in zip(["Maquina", "Turno", "Inicio Habitual", "Cierre Habitual", "Apertura Neta", "No Registrado"], [w_maq, w_turno, w_inicio, w_cierre, w_apertura, w_noregis]): pdf.cell(ancho, 6, titulo, border=1, fill=True, align='C')
        pdf.ln(); pdf.set_text_color(*COLOR_NEGRO); pdf.set_font("Arial", '', 8); pdf.set_draw_color(210, 210, 210)

        fill_row = False
        for _, fila in df_turnos.iterrows():
            pdf.set_fill_color(248, 248, 248) if fill_row else pdf.set_fill_color(255, 255, 255)
            pdf.cell(w_maq, 5.5, str(fila['MAQUINA']), border=1, fill=True, align='L'); pdf.cell(w_turno, 5.5, str(fila['TURNO']), border=1, fill=True, align='C'); pdf.cell(w_inicio, 5.5, str(fila['TXT_INICIO']), border=1, fill=True, align='C'); pdf.cell(w_cierre, 5.5, str(fila['TXT_CIERRE']), border=1, fill=True, align='C'); pdf.cell(w_apertura, 5.5, str(fila['TXT_APERTURA_NETA']), border=1, fill=True, align='C'); pdf.cell(w_noregis, 5.5, str(fila['TXT_NO_REGISTRADO']), border=1, fill=True, align='C'); pdf.ln()
            fill_row = not fill_row

    pdf.ln(10); pdf.set_font("Arial", 'B', 12); pdf.set_text_color(*COLOR_TITULO)
    pdf.cell(0, 8, txt="4. Resumen Tiempos Grupo", ln=True)
    pdf.set_draw_color(*COLOR_TITULO); pdf.line(10, pdf.get_y(), 200, pdf.get_y()); pdf.ln(4)

    w_cat = 190 / 5
    pdf.set_fill_color(211, 84, 0); pdf.set_text_color(255, 255, 255); pdf.set_draw_color(255, 255, 255); pdf.set_font("Arial", 'B', 8.5)
    for titulo in ["Produccion", "Fallas/Gestion", "Paradas Prog.", "Proyecto", "Descansos"]: pdf.cell(w_cat, 6, titulo, border=1, fill=True, align='C')
    pdf.ln(); pdf.set_text_color(*COLOR_NEGRO); pdf.set_font("Arial", '', 8); pdf.set_draw_color(210, 210, 210); pdf.set_fill_color(255, 253, 208)

    for clave in ["Produccion", "Fallas/Gestion", "Paradas Prog.", "Proyecto", "Descansos"]: pdf.cell(w_cat, 5.5, min_a_hhmm_hs(tiempos_agrupados[clave]), border=1, fill=True, align='C')
    pdf.ln()

    # --- PÁGINA 3 ---
    pdf.add_page(); pdf.set_font("Arial", 'B', 12); pdf.set_text_color(*COLOR_TITULO)
    pdf.cell(0, 8, txt="5. Análisis de Fallas y Tendencias", ln=True)
    pdf.set_draw_color(*COLOR_TITULO); pdf.line(10, pdf.get_y(), 200, pdf.get_y()); pdf.ln(4)

    pdf.set_font("Arial", 'B', 10); pdf.set_text_color(211, 84, 0)
    y_actual = pdf.get_y()
    pdf.set_xy(10, y_actual); pdf.cell(90, 6, txt="> Top 15 Fallas:", ln=0)
    pdf.set_xy(105, y_actual); pdf.cell(90, 6, txt="> Tendencia Fallas:", ln=1)

    y_graficos = pdf.get_y()
    if os.path.exists('grafico_top_fallas.png'): pdf.image('grafico_top_fallas.png', x=10, y=y_graficos, w=92)
    if os.path.exists('grafico_tendencia.png'): pdf.image('grafico_tendencia.png', x=105, y=y_graficos, w=92)

    # --- SECCIÓN 6 ---
    pdf.set_y(y_graficos + 70); pdf.ln(5)
    pdf.set_font("Arial", 'B', 12); pdf.set_text_color(*COLOR_TITULO)
    pdf.cell(0, 8, txt="6. Cuadro Resumen por Máquinas", ln=True)
    pdf.set_draw_color(*COLOR_TITULO); pdf.line(10, pdf.get_y(), 200, pdf.get_y()); pdf.ln(4)

    cols_w = [26, 22, 18, 22, 20, 22, 60]
    headers = ["MAQUINA", "PRODUCCION", "FALLAS", "PARADA PROG.", "DESCANSO", "TIEMPO NO REG.", "TOP 3 FALLAS"]

    pdf.set_fill_color(211, 84, 0); pdf.set_text_color(255, 255, 255); pdf.set_draw_color(255, 255, 255); pdf.set_font("Arial", 'B', 7)
    for w, h in zip(cols_w, headers): pdf.cell(w, 6, h, border=1, fill=True, align='C')
    pdf.ln(); pdf.set_text_color(*COLOR_NEGRO); pdf.set_font("Arial", '', 7); pdf.set_draw_color(210, 210, 210)
    fill_row = False

    for m in sorted(list(resumen_maq_dict.keys())):
        if pdf.get_y() > 250:
            pdf.add_page(); pdf.set_fill_color(211, 84, 0); pdf.set_text_color(255, 255, 255); pdf.set_draw_color(255, 255, 255); pdf.set_font("Arial", 'B', 7)
            for w, h in zip(cols_w, headers): pdf.cell(w, 6, h, border=1, fill=True, align='C')
            pdf.ln(); pdf.set_text_color(*COLOR_NEGRO); pdf.set_font("Arial", '', 7); pdf.set_draw_color(210, 210, 210)

        pdf.set_fill_color(245, 248, 250) if fill_row else pdf.set_fill_color(255, 255, 255)

        prod = min_a_hhmm_hs(resumen_maq_dict[m].get('Produccion', 0)); fallas = min_a_hhmm_hs(resumen_maq_dict[m].get('Fallas', 0))
        parada = min_a_hhmm_hs(resumen_maq_dict[m].get('ParadaProg', 0)); descanso = min_a_hhmm_hs(resumen_maq_dict[m].get('Descanso', 0))
        noreg = min_a_hhmm_hs(no_reg_dict.get(m, 0)) if no_reg_dict.get(m, 0) > 0 else "-"
        top3_str = "\n".join(top3_dict.get(m, [])) if top3_dict.get(m, []) else "-"

        lines = len(top3_dict.get(m, [])) if len(top3_dict.get(m, [])) > 1 else 1
        line_height = 4.5; row_height = lines * line_height
        x_start, y_start = pdf.get_x(), pdf.get_y()

        pdf.cell(cols_w[0], row_height, m, border=1, fill=True, align='C')
        pdf.cell(cols_w[1], row_height, prod, border=1, fill=True, align='C')
        pdf.cell(cols_w[2], row_height, fallas, border=1, fill=True, align='C')
        pdf.cell(cols_w[3], row_height, parada, border=1, fill=True, align='C')
        pdf.cell(cols_w[4], row_height, descanso, border=1, fill=True, align='C')
        pdf.cell(cols_w[5], row_height, noreg, border=1, fill=True, align='C')
        x_multi = pdf.get_x()
        pdf.multi_cell(cols_w[6], line_height, top3_str, border=1, fill=True, align='L')
        pdf.set_xy(x_start, y_start + row_height); fill_row = not fill_row

    # --- SECCIÓN 7 ---
    pdf.ln(10)
    if pdf.get_y() > 230: pdf.add_page()

    pdf.set_font("Arial", 'B', 12); pdf.set_text_color(*COLOR_TITULO)
    pdf.cell(0, 8, txt="7. Distribución de Down Time por Área", ln=True)
    pdf.set_draw_color(*COLOR_TITULO); pdf.line(10, pdf.get_y(), 200, pdf.get_y()); pdf.ln(4)

    TARGETS_DT = {'MANTEN.': {'TRG': 1.0, 'MIN': 2.0}, 'DISPOSIT.': {'TRG': 3.0, 'MIN': 4.5}, 'TECNOL.': {'TRG': 3.0, 'MIN': 4.5}, 'LOGIST.': {'TRG': 1.5, 'MIN': 2.5}, 'GESTION': {'TRG': 1.5, 'MIN': 2.0}, 'OTROS': {'TRG': 99.0, 'MIN': 99.0}}
    cols_dt_w = [30, 26, 26, 26, 26, 26, 26]; headers_dt = ["MAQUINA", "MANTEN.", "DISPOSIT.", "TECNOL.", "LOGIST.", "GESTION", "OTROS"]

    pdf.set_fill_color(211, 84, 0); pdf.set_text_color(255, 255, 255); pdf.set_draw_color(255, 255, 255); pdf.set_font("Arial", 'B', 8)
    x_start, y_start = pdf.get_x(), pdf.get_y()
    pdf.multi_cell(cols_dt_w[0], 6, "MAQUINA\nTarget vs Real", border=1, fill=True, align='C')

    for i, col_name in enumerate(headers_dt[1:]):
        x_col = x_start + cols_dt_w[0] + sum(cols_dt_w[1:i+1]); pdf.set_xy(x_col, y_start)
        if col_name == 'OTROS': pdf.cell(cols_dt_w[i+1], 12, col_name, border=1, fill=True, align='C')
        else: pdf.multi_cell(cols_dt_w[i+1], 4, f"{col_name}\nTRG: {TARGETS_DT[col_name]['TRG']}%\nMIN: {TARGETS_DT[col_name]['MIN']}%", border=1, fill=True, align='C')

    pdf.set_xy(10, y_start + 12); pdf.set_font("Arial", 'B', 8); pdf.set_draw_color(210, 210, 210)
    fill_row = False
    for m in sorted(list(areas_dt_dict.keys())):
        pdf.set_fill_color(245, 248, 250) if fill_row else pdf.set_fill_color(255, 255, 255)
        apertura_total = apertura_mensual_dict.get(m, 0)
        pdf.set_text_color(*COLOR_NEGRO); pdf.cell(cols_dt_w[0], 6, m, border=1, fill=True, align='C')
        for i, col_name in enumerate(headers_dt[1:]):
            mins_area = areas_dt_dict[m].get(col_name, 0)
            porcentaje = (mins_area / apertura_total * 100) if apertura_total > 0 else 0.0
            trg, min_lim = TARGETS_DT[col_name]['TRG'], TARGETS_DT[col_name]['MIN']
            if col_name == 'OTROS' or porcentaje == 0: pdf.set_text_color(50, 50, 50)
            elif porcentaje <= trg: pdf.set_text_color(40, 167, 69)
            elif porcentaje >= min_lim: pdf.set_text_color(220, 53, 69)
            else: pdf.set_text_color(230, 126, 34)
            pdf.cell(cols_dt_w[i+1], 6, f"{porcentaje:.1f}% ({int(mins_area)}m)", border=1, fill=True, align='C')
        pdf.ln(); fill_row = not fill_row

    # --- SECCIÓN 8: DESGLOSE DE PRODUCCIÓN ---
    pdf.ln(10)
    if pdf.get_y() > 240: pdf.add_page()

    pdf.set_font("Arial", 'B', 12); pdf.set_text_color(*COLOR_TITULO)
    pdf.cell(0, 8, txt="8. Desglose de Producción", ln=True)
    pdf.set_draw_color(*COLOR_TITULO); pdf.line(10, pdf.get_y(), 200, pdf.get_y()); pdf.ln(4)

    if not df_top_prod.empty:
        maquinas_prod = df_top_prod['MAQUINA'].unique()
        for maq in sorted(maquinas_prod):
            # Salto de página preventivo si la tabla no entra en la hoja
            if pdf.get_y() > 250: pdf.add_page()

            # Filtrar productos para esta máquina y calcular el total de piezas de su Top 5
            df_maq_prod = df_top_prod[df_top_prod['MAQUINA'] == maq]
            total_pzs = df_maq_prod['TOTAL_PRODUCIDO'].sum()

            pdf.set_font("Arial", 'B', 10); pdf.set_text_color(*COLOR_TITULO)
            pdf.cell(0, 6, txt=f"Top 5 Códigos Producidos - {maq} (Total: {int(total_pzs)} pzs)", ln=True)

            # Encabezado de la tabla de productos
            cols_prod_w = [100, 30, 30, 30]
            headers_prod = ["Codigo Producto", "Buenas", "Retrab.", "Observ."]
            pdf.set_fill_color(211, 84, 0); pdf.set_text_color(255, 255, 255); pdf.set_draw_color(255, 255, 255); pdf.set_font("Arial", 'B', 9)
            for w, h in zip(cols_prod_w, headers_prod): pdf.cell(w, 6, h, border=1, fill=True, align='C')
            pdf.ln()

            # Filas de productos
            pdf.set_text_color(*COLOR_NEGRO); pdf.set_font("Arial", '', 8); pdf.set_draw_color(210, 210, 210)
            fill_row = False
            for _, row_prod in df_maq_prod.iterrows():
                pdf.set_fill_color(245, 248, 250) if fill_row else pdf.set_fill_color(255, 255, 255)
                pdf.cell(cols_prod_w[0], 5.5, str(row_prod['CODIGO_PRODUCTO']), border=1, fill=True, align='C')
                pdf.cell(cols_prod_w[1], 5.5, str(int(row_prod['BUENAS'])), border=1, fill=True, align='C')
                pdf.cell(cols_prod_w[2], 5.5, str(int(row_prod['RETRABAJO'])), border=1, fill=True, align='C')
                pdf.cell(cols_prod_w[3], 5.5, str(int(row_prod['OBSERVADAS'])), border=1, fill=True, align='C')
                pdf.ln()
                fill_row = not fill_row
            pdf.ln(5)

    nombre_pdf = f"Reporte_CELDA_RENAULT_{FECHA_INICIO}_al_{FECHA_FIN}.pdf"

    # --- SECCIÓN 9: ANEXO DE APERTURAS Y CIERRES DIARIOS EXACTOS ---
    if not df_horarios_diarios.empty:
            pdf.add_page()
            pdf.set_font("Arial", 'B', 12)
            pdf.set_text_color(*COLOR_TITULO)
            pdf.cell(0, 8, txt="9. Anexo: Registro Diario Exacto de Apertura y Cierre", ln=True)
            pdf.set_draw_color(*COLOR_TITULO)
            pdf.line(10, pdf.get_y(), 200, pdf.get_y())
            pdf.ln(4)

            # Identificamos los turnos que existan en el período (A, B, C, etc.)
            turnos_unicos = sorted(df_horarios_diarios['TURNO'].dropna().unique())

            # Como separamos por turno en el título, quitamos la columna "Turno" y agregamos "Cierre"
            cols_anexo_w = [25, 75, 45, 45]
            headers_anexo = ["FECHA", "MAQUINA", "HORA EXACTA INICIO", "HORA EXACTA FIN"]

            for turno in turnos_unicos:
                # Subtítulo para separar visualmente cada turno
                pdf.set_font("Arial", 'B', 11)
                pdf.set_text_color(*COLOR_TITULO)
                pdf.cell(0, 8, txt=f">> TABLA DEL TURNO {turno}", ln=True)

                # Encabezado de la tabla para este turno en particular
                pdf.set_fill_color(211, 84, 0)
                pdf.set_text_color(255, 255, 255)
                pdf.set_draw_color(255, 255, 255)
                pdf.set_font("Arial", 'B', 9)

                for w, h in zip(cols_anexo_w, headers_anexo):
                    pdf.cell(w, 6, h, border=1, fill=True, align='C')
                pdf.ln()

                pdf.set_text_color(*COLOR_NEGRO)
                pdf.set_font("Arial", '', 8)
                pdf.set_draw_color(210, 210, 210)

                # Filtramos solo los datos de este turno y ordenamos cronológicamente
                df_turno = df_horarios_diarios[df_horarios_diarios['TURNO'] == turno].sort_values(by=['FECHA', 'MAQUINA'])
                fill_row = False

                for _, row in df_turno.iterrows():
                    # Salto automático si la tabla del turno es muy larga
                    if pdf.get_y() > 260:
                        pdf.add_page()
                        pdf.set_font("Arial", 'B', 10); pdf.set_text_color(*COLOR_TITULO)
                        pdf.cell(0, 8, txt=f">> TABLA DEL TURNO {turno} (Continuación)", ln=True)
                        pdf.set_fill_color(211, 84, 0); pdf.set_text_color(255, 255, 255); pdf.set_font("Arial", 'B', 9)
                        for w, h in zip(cols_anexo_w, headers_anexo): pdf.cell(w, 6, h, border=1, fill=True, align='C')
                        pdf.ln()
                        pdf.set_text_color(*COLOR_NEGRO); pdf.set_font("Arial", '', 8); pdf.set_draw_color(210, 210, 210)

                    pdf.set_fill_color(245, 248, 250) if fill_row else pdf.set_fill_color(255, 255, 255)

                    fecha_str = row['FECHA'].strftime("%d/%m/%Y")
                    maq_str = formatear_nombre_maq(row['MAQUINA'])
                    hora_inicio_str = row['HORA_INICIO'].strftime("%H:%M:%S") if not pd.isna(row['HORA_INICIO']) else "Sin Registro"
                    hora_cierre_str = row['HORA_CIERRE'].strftime("%H:%M:%S") if not pd.isna(row['HORA_CIERRE']) else "Sin Registro"

                    pdf.cell(cols_anexo_w[0], 5.5, fecha_str, border=1, fill=True, align='C')
                    pdf.cell(cols_anexo_w[1], 5.5, maq_str, border=1, fill=True, align='L')
                    pdf.cell(cols_anexo_w[2], 5.5, hora_inicio_str, border=1, fill=True, align='C')
                    pdf.cell(cols_anexo_w[3], 5.5, hora_cierre_str, border=1, fill=True, align='C')
                    pdf.ln()

                    fill_row = not fill_row

                # Espacio visual antes de dibujar la tabla del siguiente turno
                pdf.ln(8)

    pdf.output(nombre_pdf)
    print(f"¡PDF Generado exitosamente con la Sección Final! Descárgalo aquí: {nombre_pdf}")

except Exception as e:
    print("❌ Error crítico procesando la base de datos SQL.")
    print(f"Detalle técnico: {e}")

Conectando a Wiidem para extraer datos del 2026-10-13 al 2026-10-13...
✅ Datos extraídos exitosamente. Procesando y generando reportes...
¡PDF Generado exitosamente con la Sección Final! Descárgalo aquí: Reporte_CELDA_RENAULT_2026-10-13_al_2026-10-13.pdf


In [ ]:
import pandas as pd
from google.colab import userdata
import pymssql
import warnings

warnings.filterwarnings('ignore', category=UserWarning)

try:
    print("Buscando TODAS las áreas de ineficiencia...")
    conexion = pymssql.connect(
        server='200.49.98.250',
        user=userdata.get('WIIDEM_USER'),
        password=userdata.get('WIIDEM_PASS'),
        database=userdata.get('WIIDEM_DB'),
        login_timeout=15
    )

    # Buscamos todas las áreas excluyendo lo productivo o programado
    query = """
    SELECT
        COALESCE(et1.Area, 'SIN ÁREA ASIGNADA') AS Area_Nombre,
        et1.Name AS Categoria_Padre,
        SUM(e.Interval) AS MinutosTotales
    FROM EVENT_01 e
    LEFT JOIN EventType et1 ON e.EventTypeLevel1 = et1.EventTypeId
    WHERE e.Started >= '2026-10-01' AND e.Started <= '2026-10-31 23:59:59'
      AND e.Cell LIKE 'CEL%'
      AND et1.Name NOT LIKE '%PRODUCCION%'
      AND et1.Name NOT LIKE '%PARADA PROGRAMADA%'
    GROUP BY et1.Area, et1.Name
    ORDER BY MinutosTotales DESC
    """
    df = pd.read_sql(query, conexion)

    print("✅ MAPEO DE ÁREAS:")
    print(df.to_string(index=False))

    conexion.close()

except Exception as e:
    print("❌ Error:", e)

Buscando TODAS las áreas de ineficiencia...
✅ MAPEO DE ÁREAS:
      Area_Nombre Categoria_Padre  MinutosTotales
    Mantenimiento          FALLAS         5929.30
          Gestion         GESTION         1658.92
SIN ÁREA ASIGNADA        PROYECTO            0.08
